# E4B NLA — Block 1 (Sweep + Extraction) → Block 2 (Train + Eval)

Run on T4 GPU. Set `HF_TOKEN` in Colab Secrets (🔑) before starting.

**Block 1:** Layer ceiling sweep → activation extraction at best layer  
**Block 2:** ARM A training (domain-aware InfoNCE) → eval suite

In [ ]:
# ── 0. Install deps ──────────────────────────────────────────────────────────
!pip install -q transformers>=4.40 peft>=0.10 bitsandbytes>=0.43 accelerate \
    sentence-transformers scikit-learn pyarrow tqdm

In [ ]:
# ── 1. Auth + env ─────────────────────────────────────────────────────────────
import os
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN

# Clone scripts + data
!git clone https://github.com/SolshineCode/e4b-nla-colab.git /content/e4b
%cd /content/e4b
!ls

In [ ]:
# ── 2. Download Gemma-4-E4B ───────────────────────────────────────────────────
# ~18 GB; takes ~10-15 min on Colab's network
from huggingface_hub import snapshot_download
MODEL_ID = 'google/gemma-4-E4B'
LOCAL_MODEL = '/content/models/gemma-4-E4B'
os.makedirs(LOCAL_MODEL, exist_ok=True)
snapshot_download(repo_id=MODEL_ID, local_dir=LOCAL_MODEL, token=HF_TOKEN)
print('Download complete.')

In [ ]:
# ── 3. BLOCK 1a: Layer ceiling sweep ─────────────────────────────────────────
# Sweeps 6 depth fractions of 42 layers; picks highest doc_top1 mid-depth layer.
# Output: results/e4b/layer_ceiling_sweep_e4b.json
!python -u eval_layer_ceiling_sweep_e4b.py \
    --base-model {LOCAL_MODEL} \
    --n-layers 42 \
    --d-model 2560 \
    --n-perm 2000

In [ ]:
# ── 3b. Read sweep result → set LAYER_E4B ────────────────────────────────────
import json
sweep = json.load(open('results/e4b/layer_ceiling_sweep_e4b.json'))
LAYER_E4B = sweep['recommended_layer']
print(f'LAYER_E4B = {LAYER_E4B}')
print(f'Reason: {sweep["recommended_layer_reason"]}')
print(f'Gate: {sweep["gate_message"]}')
assert sweep['gate_pass'], 'Gate fail — all doc_top1 < 0.40. Stop and investigate.'

In [ ]:
# ── 4. BLOCK 1b: Extract activations (train) ─────────────────────────────────
# 653 text rows (null-text rows logged/skipped). Checkpoints every 200 rows.
# --resume safe to re-run if interrupted.
!python -u stage0_reextract_e4b.py \
    --source data/stage3_balanced/e4b_textjoin_train.parquet \
    --output data/stage3_balanced/av_sft_balanced_e4b.parquet \
    --base-model {LOCAL_MODEL} \
    --layer {LAYER_E4B} --d-model 2560 \
    --extra-positions 4 --resume

In [ ]:
# ── 5. Extract activations (eval) ─────────────────────────────────────────────
!python -u stage0_reextract_e4b.py \
    --source data/stage3_balanced/e4b_textjoin_eval.parquet \
    --output data/stage3_balanced/balanced_eval_e4b.parquet \
    --base-model {LOCAL_MODEL} \
    --layer {LAYER_E4B} --d-model 2560 \
    --resume

In [ ]:
# ── 6. Extract activations (legacy eval cmp) ──────────────────────────────────
!python -u stage0_reextract_e4b.py \
    --source data/stage3_v0_4_fineweb/e4b_textjoin_evalcmp.parquet \
    --output data/stage3_v0_4_fineweb/indomain_eval_cmp_e4b.parquet \
    --base-model {LOCAL_MODEL} \
    --layer {LAYER_E4B} --d-model 2560 \
    --resume

In [ ]:
# ── 7. BLOCK 2a: VRAM smoke (1-step real objective) ───────────────────────────
# Verifies peak VRAM < 14 GB on T4 with real ARM A objective.
# Hard fail if OOM — check neg-chunk param.
import subprocess, sys
r = subprocess.run([sys.executable, 'train_av_e4b.py',
    '--base-model', LOCAL_MODEL,
    '--data', 'data/stage3_balanced/av_sft_balanced_e4b.parquet',
    '--d-model', '2560', '--inject-layer', 'embed',
    '--contrastive', '--contrastive-domain-aware', '--contrastive-negs', '32',
    '--max-steps', '1', '--smoke-vram',
    '--output', 'checkpoints/smoke_e4b'], capture_output=False)
assert r.returncode == 0, 'VRAM smoke FAILED'

In [ ]:
# ── 8. BLOCK 2b: Main ARM A run (1500 steps) ──────────────────────────────────
# Kill gate @ step 500: check loss trend + n_unique before continuing.
# Seed 17, domain-aware InfoNCE β=1 K=2, mean-centered injection.
!python -u train_av_e4b.py \
    --base-model {LOCAL_MODEL} \
    --data data/stage3_balanced/av_sft_balanced_e4b.parquet \
    --d-model 2560 --inject-layer embed \
    --contrastive --contrastive-domain-aware \
    --contrastive-negs 32 --contrastive-beta 1.0 \
    --mean-center \
    --seed 17 --max-steps 1500 \
    --save-every 250 \
    --output checkpoints/av_e4b_armA

In [ ]:
# ── 8b. Kill-gate check @ step 500 ────────────────────────────────────────────
# Run gen_compare on step_000500 checkpoint before continuing to 1500.
!python gen_compare_e4b.py \
    checkpoints/av_e4b_armA/step_000500 \
    --eval data/stage3_balanced/balanced_eval_e4b.parquet \
    --base-model {LOCAL_MODEL}

In [ ]:
# ── 9. BLOCK 2c: Domain tracking eval ─────────────────────────────────────────
import glob
ckpts = sorted(glob.glob('checkpoints/av_e4b_armA/step_*'))
print('Checkpoints to eval:', ckpts)
for ckpt in ckpts:
    tag = ckpt.split('/')[-1]
    !python eval_domain_tracking_e4b.py {ckpt} \
        --eval data/stage3_balanced/balanced_eval_e4b.parquet \
        --train data/stage3_balanced/av_sft_balanced_e4b.parquet \
        --tag e4b_armA_{tag}

In [ ]:
# ── 10. Checkpoint eval (retrieval + 2AFC) ────────────────────────────────────
for ckpt in ckpts:
    tag = ckpt.split('/')[-1]
    !python eval_av_checkpoint_e4b.py {ckpt} \
        --eval data/stage3_balanced/balanced_eval_e4b.parquet \
        --base-model {LOCAL_MODEL} \
        --tag e4b_armA_{tag}

In [ ]:
# ── 11. Save results to Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import shutil
dest = '/content/drive/MyDrive/e4b_nla_results'
shutil.copytree('results', f'{dest}/results', dirs_exist_ok=True)
shutil.copytree('checkpoints', f'{dest}/checkpoints', dirs_exist_ok=True)
shutil.copytree('data/stage3_balanced', f'{dest}/data', dirs_exist_ok=True)
print('Saved to Drive:', dest)